In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import time

def simulate_dataset(n, scenario_params, rng):
    """
    Simulate one dataset.

    Parameters
    ----------
    n : int
        Sample size.
    scenario_params : dict
        Dictionary with keys such as:
        - gamma0, gamma1, delta
        - beta0, beta1, beta2, tau
        - sigma_y
        - confounding (bool), heterogeneity (bool)
        - a0, a1, a2, sigma_u, gamma_u  (if confounding)
        - tau0, tau1                     (if heterogeneity)
    rng : np.random.Generator
        Numpy random number generator.

    Returns
    -------
    df : pandas.DataFrame
        Columns: Y, D, X, Z, G
    """

    # Running variable and cutoff
    X = rng.uniform(-1, 1, size=n)
    c = 0.0
    Z = (X >= c).astype(int)  # cutoff indicator / instrument

    # Compliance mechanism
    gamma0 = scenario_params["gamma0"]
    gamma1 = scenario_params["gamma1"]
    delta  = scenario_params["delta"]

    g = 1 / (1 + np.exp(-(gamma0 + gamma1 * X)))   # smooth baseline
    p_d = g + delta * Z                            # add discontinuity at cutoff
    p_d = np.clip(p_d, 1e-4, 1 - 1e-4)             # avoid 0/1 probabilities

    D = rng.binomial(1, p_d)

    # Baseline outcome surface m(X)
    beta0 = scenario_params["beta0"]
    beta1 = scenario_params["beta1"]
    beta2 = scenario_params["beta2"]
    tau   = scenario_params["tau"]  # reference effect (e.g., for homogeneous cases)

    m = beta0 + beta1 * X + beta2 * X**2
    eps = rng.normal(0, scenario_params["sigma_y"], size=n)

    # Build treatment effect vector (allows heterogeneity)
    if scenario_params.get("heterogeneity", False):
        # Binary subgroup, 0/1
        G = rng.binomial(1, 0.5, size=n)
        tau0 = scenario_params["tau0"]
        tau1 = scenario_params["tau1"]
        tau_vec = np.where(G == 1, tau1, tau0)
    else:
        # Homogeneous effect
        G = np.zeros(n, dtype=int)
        tau_vec = tau * np.ones(n)

    # Confounding branch
    if scenario_params.get("confounding", False):
        eta = rng.normal(0, scenario_params["sigma_u"], size=n)
        a0 = scenario_params["a0"]
        a1 = scenario_params["a1"]
        a2 = scenario_params["a2"]
        gamma_u = scenario_params["gamma_u"]

        U = a0 + a1 * X + a2 * X**2 + eta
        Y = tau_vec * D + m + gamma_u * U + eps
    else:
        # No additional confounder
        Y = tau_vec * D + m + eps

    # Return a consistent schema across all scenarios
    df = pd.DataFrame({
        "Y": Y,
        "D": D,
        "X": X,
        "Z": Z,
        "G": G  # will be 0 for all observations when heterogeneity=False
    })

    return df

In [2]:
scenarios = {
    "S1_baseline": {
        # First-stage / compliance
        "gamma0": 0.0,
        "gamma1": 1.0,
        "delta": 0.25,      # medium first stage

        # Outcome surface m(X) = beta0 + beta1 X + beta2 X^2
        "beta0": 0.0,
        "beta1": 0.5,
        "beta2": 0.5,       # mild curvature

        # Treatment effect and noise
        "tau": 1.0,         # homogeneous effect
        "sigma_y": 1.0,

        # Scenario flags
        "confounding": False,
        "heterogeneity": False
    },

    "S2_curvature": {
        "gamma0": 0.0,
        "gamma1": 1.0,
        "delta": 0.25,      # same first stage as S1

        "beta0": 0.0,
        "beta1": 0.5,
        "beta2": 2.0,       # stronger curvature to stress bandwidth / misspecification

        "tau": 1.0,
        "sigma_y": 1.0,

        "confounding": False,
        "heterogeneity": False
    },

    "S3_weakFS": {
        "gamma0": 0.0,
        "gamma1": 1.0,
        "delta": 0.10,      # weak first stage

        "beta0": 0.0,
        "beta1": 0.5,
        "beta2": 0.5,

        "tau": 1.0,
        "sigma_y": 1.0,

        "confounding": False,
        "heterogeneity": False
    },

    "S4_confounding": {
        "gamma0": 0.0,
        "gamma1": 1.0,
        "delta": 0.25,      # medium first stage

        "beta0": 0.0,
        "beta1": 0.5,
        "beta2": 0.5,

        "tau": 1.0,
        "sigma_y": 1.0,

        "confounding": True,
        "heterogeneity": False,

        # Smooth confounder U = a0 + a1 X + a2 X^2 + eta
        "a0": 0.0,
        "a1": 0.8,
        "a2": 0.8,
        "sigma_u": 1.0,
        "gamma_u": 0.8      # strength of confounder in outcome
    },

    "S5_heterogeneity": {
        "gamma0": 0.0,
        "gamma1": 1.0,
        "delta": 0.25,      # medium first stage

        "beta0": 0.0,
        "beta1": 0.5,
        "beta2": 0.5,

        # Group-specific effects: tau0 for G=0, tau1 for G=1
        "tau0": 0.5,
        "tau1": 1.5,
        "tau": 1.0,         # reference value for summaries (overall target)

        "sigma_y": 1.0,

        "confounding": False,
        "heterogeneity": True
    }
}

In [3]:
# Naive global OLS 
def estimate_naive_ols(df, tau_true=None):
    """
    Naïve global OLS: regress Y on D and X, ignoring the cutoff and fuzziness.

    Model: Y_i = beta0 + beta1 * D_i + beta2 * X_i + e_i
    """

    y = df["Y"].values
    X_design = np.column_stack([df["D"].values, df["X"].values])
    X_design = sm.add_constant(X_design)

    model = sm.OLS(y, X_design).fit()

    beta1 = model.params[1]
    se1   = model.bse[1]
    n     = df.shape[0]

    return {
        "estimator": "naive_ols",
        "tau_true": tau_true,
        "tau_hat": float(beta1),
        "se_tau": float(se1),
        "n": int(n)
    }

# Incorrect sharp RDD
def estimate_sharp_local(df, h, tau_true=None, cutoff=0.0):
    """
    Incorrect sharp-RDD estimator applied to a fuzzy design.

    Within |X - cutoff| <= h, fit:
        Y_i = gamma0 + gamma1 * Z_i + e_i,
    where Z_i = 1{X_i >= cutoff}.
    """

    tmp = df[np.abs(df["X"] - cutoff) <= h].copy()
    n = tmp.shape[0]

    if n == 0:
        return {
            "estimator": "sharp_local",
            "bandwidth": h,
            "tau_true": tau_true,
            "tau_hat": np.nan,
            "se_tau": np.nan,
            "n_left": 0,
            "n_right": 0,
            "n": 0
        }

    # Ensure Z exists
    if "Z" not in tmp.columns:
        tmp["Z"] = (tmp["X"] >= cutoff).astype(int)

    n_left  = int((tmp["X"] <  cutoff).sum())
    n_right = int((tmp["X"] >= cutoff).sum())

    y = tmp["Y"].values
    Z_design = sm.add_constant(tmp["Z"].values)

    model = sm.OLS(y, Z_design).fit()
    gamma1 = model.params[1]
    se1    = model.bse[1]

    return {
        "estimator": "sharp_local",
        "bandwidth": h,
        "tau_true": tau_true,
        "tau_hat": float(gamma1),
        "se_tau": float(se1),
        "n_left": n_left,
        "n_right": n_right,
        "n": int(n)
    }

# Local mean fuzzy Wald
def estimate_local_wald(df, h, tau_true=None, cutoff=0.0):
    """
    Local-mean fuzzy FRD Wald estimator using a symmetric window |X - cutoff| <= h.

        tau_hat = (jump in Y) / (jump in D)
                = (E[Y | X in [0,h]] - E[Y | X in [-h,0))) /
                  (E[D | X in [0,h]] - E[D | X in [-h,0))).
    """

    tmp = df[np.abs(df["X"] - cutoff) <= h]

    left  = tmp[tmp["X"] <  cutoff]
    right = tmp[tmp["X"] >= cutoff]

    n_left  = left.shape[0]
    n_right = right.shape[0]

    if n_left == 0 or n_right == 0:
        return {
            "estimator": "local_wald",
            "bandwidth": h,
            "tau_true": tau_true,
            "tau_hat": np.nan,
            "first_stage_jump": np.nan,
            "jump_Y": np.nan,
            "n_left": n_left,
            "n_right": n_right
        }

    y_left,  y_right  = left["Y"].mean(), right["Y"].mean()
    d_left,  d_right  = left["D"].mean(), right["D"].mean()

    jump_Y = y_right - y_left
    jump_D = d_right - d_left

    tau_hat = np.nan if jump_D == 0 else jump_Y / jump_D

    return {
        "estimator": "local_wald",
        "bandwidth": h,
        "tau_true": tau_true,
        "tau_hat": float(tau_hat),
        "first_stage_jump": float(jump_D),
        "jump_Y": float(jump_Y),
        "n_left": int(n_left),
        "n_right": int(n_right)
    }

# Local 2SLS (proper fuzzy RDD/local IV)
def estimate_local_2sls(df, h, tau_true=None, cutoff=0.0):
    """
    Local 2SLS / local IV fuzzy RDD estimator with fully interacted local-linear spec.

    Window: |X - cutoff| <= h.
    First stage:
        D_i = a0 + a1 * Z_i + a2 * Xc_i + a3 * (Z_i * Xc_i) + u_i
    Second stage:
        Y_i = g0 + g1 * D_hat_i + g2 * Xc_i + g3 * (Z_i * Xc_i) + eta_i

    Returns tau_hat = g1 and a simple first-stage F-stat on Z.
    """

    tmp = df[np.abs(df["X"] - cutoff) <= h].copy()
    n = tmp.shape[0]

    if n == 0:
        return {
            "estimator": "local_2sls",
            "bandwidth": h,
            "tau_true": tau_true,
            "tau_hat": np.nan,
            "se_tau": np.nan,
            "first_stage_F": np.nan,
            "n_left": 0,
            "n_right": 0,
            "n": 0
        }

    # Ensure Z exists
    if "Z" not in tmp.columns:
        tmp["Z"] = (tmp["X"] >= cutoff).astype(int)

    Xc = tmp["X"].values - cutoff
    Z  = tmp["Z"].values.astype(float)
    D  = tmp["D"].values
    Y  = tmp["Y"].values

    # Fully interacted local-linear controls: const, Xc, Z*Xc
    W = np.column_stack([np.ones(n), Xc, Z * Xc])   # exogenous controls
    Z_col = Z.reshape(-1, 1)

    # First stage: D ~ Z + W (so regressors: const, Xc, Z*Xc, Z)
    X_fs = np.column_stack([W, Z_col])
    fs_model = sm.OLS(D, X_fs).fit()
    D_hat = fs_model.fittedvalues

    # Extract t-stat for Z coefficient (last column) and compute F
    t_Z = fs_model.tvalues[-1]
    fs_F = float(t_Z ** 2)

    # Second stage: Y ~ D_hat + W (regressors: const, Xc, Z*Xc, D_hat)
    X_ss = np.column_stack([W, D_hat])
    ss_model = sm.OLS(Y, X_ss).fit()

    tau_hat = ss_model.params[-1]   # coefficient on D_hat
    se_tau  = ss_model.bse[-1]

    n_left  = int((tmp["X"] <  cutoff).sum())
    n_right = int((tmp["X"] >= cutoff).sum())

    return {
        "estimator": "local_2sls",
        "bandwidth": h,
        "tau_true": tau_true,
        "tau_hat": float(tau_hat),
        "se_tau": float(se_tau),
        "first_stage_F": fs_F,
        "n_left": n_left,
        "n_right": n_right,
        "n": int(n)
    }


In [4]:
# Reproducibility
rng = np.random.default_rng(123456)


start_time = time.time()

# Monte Carlo settings
B = 200
n_grid = [2000, 10000]   # two sample sizes
bandwidths = [0.05, 0.10, 0.20]

all_results = []  # list of dicts from estimators (small enough to keep)

for n in n_grid:

    # ---- DGP output file for this n ----
    dgp_path = f"frd_dgp_n{n}.csv"
    wrote_header = False  # to control header writing on first append

    for scen_name, scen_params in scenarios.items():
        tau_ref = scen_params["tau"]

        for b in range(1, B + 1):

            # 1) Simulate one dataset
            df = simulate_dataset(n, scen_params, rng)

            # Tag with sample size, scenario, and replication id
            df["n"] = n
            df["scenario"] = scen_name
            df["rep"] = b

            # 1a) Append DGP directly to disk (avoid huge in-memory concat)
            df.to_csv(
                dgp_path,
                mode="a",
                header=(not wrote_header),
                index=False
            )
            wrote_header = True

            # 2) Run estimators in Python (pooled)
            res_naive = estimate_naive_ols(df, tau_true=tau_ref)
            res_naive.update({
                "scenario": scen_name,
                "rep": b,
                "n": n,
                "h": np.nan,
                "group": "pooled"
            })
            all_results.append(res_naive)

            for h in bandwidths:
                res_sharp = estimate_sharp_local(df, h, tau_true=tau_ref)
                res_sharp.update({
                    "scenario": scen_name,
                    "rep": b,
                    "n": n,
                    "h": h,
                    "group": "pooled"
                })
                all_results.append(res_sharp)

                res_wald = estimate_local_wald(df, h, tau_true=tau_ref)
                res_wald.update({
                    "scenario": scen_name,
                    "rep": b,
                    "n": n,
                    "h": h,
                    "group": "pooled"
                })
                all_results.append(res_wald)

                res_2sls = estimate_local_2sls(df, h, tau_true=tau_ref)
                res_2sls.update({
                    "scenario": scen_name,
                    "rep": b,
                    "n": n,
                    "h": h,
                    "group": "pooled"
                })
                all_results.append(res_2sls)

            # 3) Subgroup analysis in S5_heterogeneity
            if scen_name == "S5_heterogeneity":
                for group_label, df_sub in {
                    "G0": df[df["G"] == 0],
                    "G1": df[df["G"] == 1],
                }.items():
                    if df_sub.empty:
                        continue

                    tau_true_group = scen_params["tau0"] if group_label == "G0" else scen_params["tau1"]

                    res_naive_g = estimate_naive_ols(df_sub, tau_true=tau_true_group)
                    res_naive_g.update({
                        "scenario": scen_name,
                        "rep": b,
                        "n": n,
                        "h": np.nan,
                        "group": group_label
                    })
                    all_results.append(res_naive_g)

                    for h in bandwidths:
                        res_sharp_g = estimate_sharp_local(df_sub, h, tau_true=tau_true_group)
                        res_sharp_g.update({
                            "scenario": scen_name,
                            "rep": b,
                            "n": n,
                            "h": h,
                            "group": group_label
                        })
                        all_results.append(res_sharp_g)

                        res_wald_g = estimate_local_wald(df_sub, h, tau_true=tau_true_group)
                        res_wald_g.update({
                            "scenario": scen_name,
                            "rep": b,
                            "n": n,
                            "h": h,
                            "group": group_label
                        })
                        all_results.append(res_wald_g)

                        res_2sls_g = estimate_local_2sls(df_sub, h, tau_true=tau_true_group)
                        res_2sls_g.update({
                            "scenario": scen_name,
                            "rep": b,
                            "n": n,
                            "h": h,
                            "group": group_label
                        })
                        all_results.append(res_2sls_g)

# 4) Write Python-estimator results once (all n pooled)
results_df = pd.DataFrame(all_results)
results_df.to_csv("frd_sim_results_python.csv", index=False)

end_time = time.time()
elapsed = end_time - start_time
print(f"Total time: {elapsed/60:.2f} minutes")

print("Wrote DGP files:")
for n in n_grid:
    print(f"  frd_dgp_n{n}.csv")


Total time: 0.72 minutes
Wrote DGP files:
  frd_dgp_n2000.csv
  frd_dgp_n10000.csv


In [5]:
results_df = pd.read_csv("frd_sim_results_python.csv")
print(results_df.head(n=10))
results_df.groupby(["scenario", "estimator"]).size()

     estimator  tau_true   tau_hat    se_tau     n     scenario  rep     h  \
0    naive_ols       1.0  1.024192  0.054430  2000  S1_baseline    1   NaN   
1  sharp_local       1.0 -0.184292  0.217573  2000  S1_baseline    1  0.05   
2   local_wald       1.0 -2.263192       NaN  2000  S1_baseline    1  0.05   
3   local_2sls       1.0  3.858402  2.231245  2000  S1_baseline    1  0.05   
4  sharp_local       1.0  0.309333  0.156555  2000  S1_baseline    1  0.10   
5   local_wald       1.0  1.127477       NaN  2000  S1_baseline    1  0.10   
6   local_2sls       1.0  5.148017  2.454602  2000  S1_baseline    1  0.10   
7  sharp_local       1.0  0.401953  0.111822  2000  S1_baseline    1  0.20   
8   local_wald       1.0  1.455138       NaN  2000  S1_baseline    1  0.20   
9   local_2sls       1.0  0.190496  1.235186  2000  S1_baseline    1  0.20   

    group  bandwidth  n_left  n_right  first_stage_jump    jump_Y  \
0  pooled        NaN     NaN      NaN               NaN       NaN   
1  

scenario          estimator  
S1_baseline       local_2sls     1200
                  local_wald     1200
                  naive_ols       400
                  sharp_local    1200
S2_curvature      local_2sls     1200
                  local_wald     1200
                  naive_ols       400
                  sharp_local    1200
S3_weakFS         local_2sls     1200
                  local_wald     1200
                  naive_ols       400
                  sharp_local    1200
S4_confounding    local_2sls     1200
                  local_wald     1200
                  naive_ols       400
                  sharp_local    1200
S5_heterogeneity  local_2sls     3600
                  local_wald     3600
                  naive_ols      1200
                  sharp_local    3600
dtype: int64

In [6]:
h_list = [0.05, 0.10, 0.20]

for scen_name, params in scenarios.items():
    true_delta = params["delta"]
    tmp = sim_raw_df[sim_raw_df["scenario"] == scen_name]

    print(f"\n{scen_name}:  true delta = {true_delta:.3f}")

    for h in h_list:
        left  = tmp[(tmp["X"] < 0) & (tmp["X"] >= -h)]
        right = tmp[(tmp["X"] >= 0) & (tmp["X"] <=  h)]

        p_left  = left["D"].mean()
        p_right = right["D"].mean()
        jump    = p_right - p_left

        n_left  = left.shape[0]
        n_right = right.shape[0]

        print(
            f"  h={h:.2f}: n_L={n_left:6d}, n_R={n_right:6d}, "
            f"mean D_L={p_left:.3f}, mean D_R={p_right:.3f}, jump={jump:.3f}"
        )

NameError: name 'sim_raw_df' is not defined

In [ ]:
import numpy as np
import pandas as pd

h_list = [0.05, 0.10, 0.20]

for scen_name, params in scenarios.items():
    delta = params["delta"]
    tau   = params["tau"]
    print(f"\n{scen_name}:  target reduced-form jump ≈ tau * delta = {tau:.2f} * {delta:.2f} = {tau*delta:.3f}")

    tmp = sim_raw_df[sim_raw_df["scenario"] == scen_name]

    for h in h_list:
        left  = tmp[(tmp["X"] < 0) & (tmp["X"] >= -h)]
        right = tmp[(tmp["X"] >= 0) & (tmp["X"] <=  h)]

        y_left  = left["Y"].mean()
        y_right = right["Y"].mean()
        jump_y  = y_right - y_left

        n_left  = left.shape[0]
        n_right = right.shape[0]

        print(
            f"  h={h:.2f}: n_L={n_left:6d}, n_R={n_right:6d}, "
            f"mean Y_L={y_left:.3f}, mean Y_R={y_right:.3f}, jump_Y={jump_y:.3f}"
        )

In [ ]:
import numpy as np
import pandas as pd

h_list = [0.05, 0.10, 0.20]

for scen_name, params in scenarios.items():
    tau_true = params["tau"]
    print(f"\n{scen_name}:  true reference tau = {tau_true:.3f}")

    tmp = sim_raw_df[sim_raw_df["scenario"] == scen_name]

    for h in h_list:
        left  = tmp[(tmp["X"] < 0) & (tmp["X"] >= -h)]
        right = tmp[(tmp["X"] >= 0) & (tmp["X"] <=  h)]

        # Reduced form and first stage
        y_left,  y_right  = left["Y"].mean(), right["Y"].mean()
        d_left,  d_right  = left["D"].mean(), right["D"].mean()
        d_jump = d_right - d_left
        y_jump = y_right - y_left

        # Naive local Wald
        tau_hat = y_jump / d_jump if d_jump != 0 else np.nan

        print(
            f"  h={h:.2f}: jump_Y={y_jump:.3f}, jump_D={d_jump:.3f}, "
            f"Wald(h)={tau_hat:.3f}"
        )

In [ ]:
import numpy as np
import pandas as pd

scen_name = "S5_heterogeneity"
params = scenarios[scen_name]

print(f"Scenario {scen_name}: tau0={params['tau0']}, tau1={params['tau1']}")

tmp = sim_raw_df[sim_raw_df["scenario"] == scen_name]

# Check G balance
print("\nG distribution in S5_heterogeneity:")
print(tmp["G"].value_counts(normalize=True))

# Choose a bandwidth for subgroup Walds
h = 0.10

for g_val in [0, 1]:
    sub = tmp[tmp["G"] == g_val]

    left  = sub[(sub["X"] < 0) & (sub["X"] >= -h)]
    right = sub[(sub["X"] >= 0) & (sub["X"] <=  h)]

    y_left,  y_right  = left["Y"].mean(), right["Y"].mean()
    d_left,  d_right  = left["D"].mean(), right["D"].mean()
    y_jump = y_right - y_left
    d_jump = d_right - d_left
    tau_hat_g = y_jump / d_jump if d_jump != 0 else np.nan

    print(
        f"\nG={g_val}: n_L={left.shape[0]}, n_R={right.shape[0]}, "
        f"jump_Y={y_jump:.3f}, jump_D={d_jump:.3f}, Wald_G={tau_hat_g:.3f}"
    )

# Overall Wald for comparison
left_all  = tmp[(tmp["X"] < 0) & (tmp["X"] >= -h)]
right_all = tmp[(tmp["X"] >= 0) & (tmp["X"] <=  h)]
y_jump_all = right_all["Y"].mean() - left_all["Y"].mean()
d_jump_all = right_all["D"].mean() - left_all["D"].mean()
tau_hat_all = y_jump_all / d_jump_all if d_jump_all != 0 else np.nan

print(f"\nOverall Wald (S5, h={h}): {tau_hat_all:.3f}")

In [ ]:
import numpy as np
import pandas as pd

# Variant of simulate_dataset that also returns U (for diagnostics)
def simulate_dataset_with_U(n, scenario_params, rng):
    X = rng.uniform(-1, 1, size=n)
    c = 0.0
    Z = (X >= c).astype(int)

    gamma0 = scenario_params["gamma0"]
    gamma1 = scenario_params["gamma1"]
    delta  = scenario_params["delta"]

    g = 1 / (1 + np.exp(-(gamma0 + gamma1 * X)))
    p_d = np.clip(g + delta * Z, 1e-4, 1 - 1e-4)
    D = rng.binomial(1, p_d)

    beta0 = scenario_params["beta0"]
    beta1 = scenario_params["beta1"]
    beta2 = scenario_params["beta2"]
    tau   = scenario_params["tau"]
    m = beta0 + beta1 * X + beta2 * X**2
    eps = rng.normal(0, scenario_params["sigma_y"], size=n)

    # Heterogeneity not needed here; focus on confounding
    tau_vec = tau * np.ones(n)

    U = np.zeros(n)
    if scenario_params.get("confounding", False):
        eta = rng.normal(0, scenario_params["sigma_u"], size=n)
        a0 = scenario_params["a0"]
        a1 = scenario_params["a1"]
        a2 = scenario_params["a2"]
        gamma_u = scenario_params["gamma_u"]
        U = a0 + a1 * X + a2 * X**2 + eta
        Y = tau_vec * D + m + gamma_u * U + eps
    else:
        Y = tau_vec * D + m + eps
        U = np.nan * np.ones(n)

    df = pd.DataFrame({"Y": Y, "D": D, "X": X, "Z": Z, "U": U})
    return df

# Large n for clear diagnostics
rng = np.random.default_rng(123)
n = 200000
params_S4 = scenarios["S4_confounding"]

df4 = simulate_dataset_with_U(n, params_S4, rng)

print("Correlations in S4_confounding:")
print(df4[["D", "Y", "U"]].corr())

# Local smoothness of U around cutoff
h = 0.10
left_U  = df4[(df4["X"] < 0) & (df4["X"] >= -h)]["U"].mean()
right_U = df4[(df4["X"] >= 0) & (df4["X"] <=  h)]["U"].mean()

print(f"\nMean U left ([-h,0)): {left_U:.3f}")
print(f"Mean U right ([0,h]): {right_U:.3f}")
print(f"Difference (right - left): {right_U - left_U:.3f}")

In [ ]:
import numpy as np
import pandas as pd

h_list = [0.05, 0.10, 0.20]

for scen_name, params in scenarios.items():
    tmp = sim_raw_df[sim_raw_df["scenario"] == scen_name].copy()

    beta0 = params["beta0"]
    beta1 = params["beta1"]
    beta2 = params["beta2"]

    # Reconstruct m(X)
    tmp["mX"] = beta0 + beta1 * tmp["X"] + beta2 * tmp["X"]**2

    print(f"\n{scen_name}: baseline m(X) smoothness check")

    for h in h_list:
        left  = tmp[(tmp["X"] < 0) & (tmp["X"] >= -h)]
        right = tmp[(tmp["X"] >= 0) & (tmp["X"] <=  h)]

        m_left  = left["mX"].mean()
        m_right = right["mX"].mean()
        diff_m  = m_right - m_left

        print(
            f"  h={h:.2f}: mean m_L={m_left:.3f}, mean m_R={m_right:.3f}, "
            f"diff={diff_m:.3f}"
        )